In [ ]:
# =========================
# Básicas
# =========================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# =========================
# Astronomía / Light curves
# =========================
from astropy.table import Table
from wotan import flatten
import lightkurve as lk
# =========================
# Modelado (binarias / exoplanetas)
# =========================
import pymc3 as pm
import pymc3_ext as pmx
import exoplanet as xo
import starry
import theano.tensor as tt

# =========================
# Visualización avanzada (opcional)
# =========================
from mpl_toolkits import mplot3d
from matplotlib import animation
from matplotlib.ticker import ScalarFormatter
from IPython.display import HTML

# =========================
# Utilidades
# =========================
from scipy.stats import median_abs_deviation
from tqdm.notebook import tqdm
from corner import corner

In [ ]:
np.random.seed(12)
starry.config.lazy = False
starry.config.quiet = True

In [ ]:

# =========================
# 1. Cargar datos
# =========================
file_path = "tic_427395136_s_6_corrected.csv"

data = pd.read_csv(file_path)
time = np.asarray(data["time"])
flux = np.asarray(data["flux"])
sigma_real = np.asarray(data["flux_err"])

# =========================
# 2. Detrending con WOTAN
# =========================
flat_flux, trend_flux = flatten(
    time,
    flux,
    window_length=0.85,
    break_tolerance=1.0,
    method="hspline",
    return_trend=True
)


# =========================
# 3. Plot original + trend
# =========================
plt.figure(figsize=(10,4))

plt.scatter(time, flux, s=5, color='k', label="Original")
plt.plot(time, trend_flux, color='r', lw=2, label="Trend")

plt.xlabel('Time MJD [days]')
plt.ylabel('Flux')
plt.legend()
plt.title("Wotan Detrending")
plt.tight_layout()
plt.show()


# =========================
# 4. Plot flatten (el importante)
# =========================
plt.figure(figsize=(10,4))

plt.scatter(time, flat_flux, s=4.5, color='b')

plt.xlabel('Time MJD [days]')
plt.ylabel('Normalised Flux')
plt.title("Flattened Light Curve (Wotan)")
plt.tight_layout()
plt.show()


# =========================
# 5. Guardar resultado
# =========================
df = pd.DataFrame({
    "time": time,
    "flat_flux": flat_flux,
    "trend_flux": trend_flux
})

df.to_csv("tic_427395136_trending.csv", index=False)

# Starry

In [ ]:
from astroquery.mast import Catalogs

tic_id = "TIC 427395136"
Label = "TIC 427395136"
cat = Catalogs.query_object(tic_id, catalog="tic")
 
cat['ID', 'Teff', 'logg', 'mass', 'rad', 'Tmag', 'GAIAmag']
r_A      = float(cat['rad'][0])#  radius in R_sun
m_A      = float(cat['mass'][0])
teff = float(cat['Teff'][0])
Tmag = float(cat["Tmag"][0])
print(r_A)
print(m_A)
print(teff)
print(Tmag)

In [ ]:
#prior

inc_A, sd_inc_A   = 87.0, 1  # inclination in degrees
amp_A, sd_amp_A   = 0.999, 0.001 # amplitude (a value prop. to luminosity)
r_A, sd_r_A       = 2.5, 0.1664 #  radius in R_sun
m_A, sd_m_A       = 0.5, 0.191  # mass in M_sun
u_A               = [0.3, 0.2],  # limb darkening coefficients

inc_B, sd_inc_B   = 85.0, 1  # inclination in degrees
amp_B, sd_amp_B   = 0.001, 0.001 # amplitude (a value prop. to luminosity)
r_B, sd_r_B       = 0.2956, 0.0568  #  radius in R_sun
m_B, sd_m_B       = 0.1, 0.0091  #  mass in M_sun
porb_B, sd_porb_B = 6.485, 0.01  # orbital period in days
t0_B, sd_t0_B     = 1486.349, 0.01  # reference time in days (when it transits A)
ecc_B, sd_ecc_B   = 0.4, 0.01
w_B, sd_w_B       = 120, 2  # longitude of pericenter in degrees
u_B               = [0.3, 0.2],  # limb darkening coefficients

prot_s            = 0

In [ ]:
t_real = time
flux_real = flat_flux
sigma_sd = 1.4826*median_abs_deviation(flux_real)

In [ ]:
fig, ax = plt.subplots(1, figsize=(12,5))
ax.plot(t_real, flux_real, ".k", alpha=0.6, ms=5)
ax.set_xlabel("time [days]", fontsize=24)
ax.set_ylabel("normalized flux", fontsize=24);

In [ ]:
map = starry.Map(ydeg=5)
A_y = np.array(map.y[1:])

map.reset()
B_y = np.array(map.y[1:])

In [ ]:
A = dict(
    ydeg   = 5,  # degree of the map
    udeg   = 2,  # degree of the limb darkening
    inc    = inc_A,  # inclination in degrees
    amp    = amp_A,  # amplitude (a value prop. to luminosity)
    r      = r_A, #  radius in R_sun
    m      = m_A,  # mass in M_sun
    prot   = prot_s,  # rotational period in days
    u      = u_A[0],  # limb darkening coefficients
    y      = A_y,  # the spherical harmonic coefficients
)
B = dict(
    ydeg   = 5,  # degree of the map
    udeg   = 2,  # degree of the limb darkening
    inc    = inc_B,  # inclination in degrees
    amp    = amp_B,  # amplitude (a value prop. to luminosity)
    r      = r_B,  #  radius in R_sun
    m      = m_B,  #  mass in M_sun
    prot   = prot_s,  # rotational period in days
    porb   = porb_B,  # orbital period in days
    t0     = t0_B,  # reference time in days (when it transits A)
    ecc    = ecc_B,
    w      = w_B,  # longitude of pericenter in degrees
    u      = u_B[0],  # limb darkening coefficients
    y      = B_y,  # the spherical harmonic coefficients
)

In [ ]:
pri = starry.Primary(
    starry.Map(ydeg=A["ydeg"], udeg=A["udeg"], inc=A["inc"], amp=A["amp"]),
    r=A["r"],
    m=A["m"],
    prot=A["prot"],
)
pri.map[1:] = A["u"]
pri.map[1:, :] = A["y"]

sec = starry.Secondary(
    starry.Map(ydeg=B["ydeg"], udeg=B["udeg"], inc=B["inc"], amp=B["amp"]),
    r=B["r"],
    m=B["m"],
    porb=B["porb"],
    prot=B["prot"],
    t0=B["t0"],
    inc=B["inc"],
    ecc=B["ecc"],
    w=B['w'],
)
sec.map[1:] = B["u"]
sec.map[1:, :] = B["y"]

In [ ]:
sys = starry.System(pri, sec)

In [ ]:
t = t_real
flux_true = sys.flux(t_real)
#sigma = sigma_real
# flux = flux_real + sigma_real * np.random.randn(len(t))
flux = flux_real + sigma_sd * np.random.randn(len(t))

In [ ]:
fig, ax = plt.subplots(1, figsize=(18, 9))
ax.plot(t_real, flux_real, "k.", alpha=0.5, ms=10)
ax.plot(t_real, flux_true, lw=2)
# ax.plot(t_real, flux_real, ".k", ms=4, color= 'k' ,label="Sector 06")
ax.set_xlabel("time [days]", fontsize=24)
ax.set_ylabel("normalized flux", fontsize=24);

# ax.set_xlim(2175 , 2181);

In [ ]:
np.savez(Label+".npz", A=A, B=B, t=t_real, flux=flux, sigma=sigma_real)

np.random.seed(12)
starry.config.lazy = True
starry.config.quiet = True

data = np.load(Label+".npz", allow_pickle=True)
A = data["A"].item()
B = data["B"].item()
t = data["t"]
flux = data["flux"]
sigma = data["sigma"]

In [ ]:
fig, ax = plt.subplots(1, figsize=(12, 5))
ax.plot(t, flux_real, "b.", alpha=0.5, ms=5)
ax.plot(t, flux, "k.", alpha=0.5, ms=5)
ax.set_xlabel("time [days]", fontsize=24)
ax.set_ylabel("normalized flux", fontsize=24);

In [ ]:
with pm.Model() as model:
    PositiveNormal = pm.Bound(pm.Normal, lower=0.0)

    # Primary
    A_inc = pm.Normal("A_inc", mu = inc_A, sd = sd_inc_A, testval = inc_A)
    A_amp = pm.Normal("A_amp", mu = amp_A, sd = sd_amp_A, testval= amp_A)
    A_r = PositiveNormal("A_r", mu = r_A, sd = sd_r_A, testval = r_A)
    A_m = PositiveNormal("A_m", mu = m_A, sd = sd_m_A, testval = m_A)
    A_prot = prot_s
    u_A = xo.distributions.QuadLimbDark("u_A", testval=np.array(A['u']))
    pri = starry.Primary(
        starry.Map(ydeg=A["ydeg"], udeg=A["udeg"], inc=A_inc, amp=A_amp),
        r=A_r,
        m=A_m,
        prot=A_prot,
    )
    pri.map[1] = A["u"][0]
    pri.map[2] = A["u"][1]

    # Secondary
    B_inc = pm.Normal("B_inc", mu = inc_B, sd = sd_inc_B, testval = inc_B)
    B_amp = pm.Normal("B_amp", mu = amp_B, sd = sd_amp_B, testval = amp_B)
    B_r = PositiveNormal("B_r", mu = r_B, sd = sd_r_B, testval = r_B)
    B_m = PositiveNormal("B_m", mu = m_B, sd = sd_m_B, testval = m_B)
    B_prot = prot_s
    B_porb = PositiveNormal("B_porb", mu = porb_B, sd = sd_porb_B, testval = porb_B)
    B_t0 = pm.Normal("B_t0", mu = t0_B, sd = sd_t0_B, testval = t0_B)
    u_B = xo.distributions.QuadLimbDark("u_B", testval=np.array(B['u']))
    B_ecc = pm.TruncatedNormal("B_ecc", mu = ecc_B, sigma = sd_ecc_B, lower = 0, upper = 1)
    #B_ecc = pm.Uniform("B_ecc", 0, 1, initval=ecc_B)
    B_w = pm.Normal("B_w", mu = w_B, sd = sd_w_B, testval = w_B)#pm.Uniform("B_w", 0, 360, initval = w_B)
    #B_Omega = pm.Uniform("B_Omega", 0, 360, initval=Omega_B)
    sec = starry.Secondary(
        starry.Map(ydeg=B["ydeg"], udeg=B["udeg"], inc=B_inc, amp=B_amp),
        r=B_r,
        m=B_m,
        porb=B_porb,
        prot=B_prot,
        t0=B_t0,
        inc=B_inc,
        ecc=B_ecc,
        w=B_w,
        #Omega=B_Omega
    )
    sec.map[1] = B["u"][0]
    sec.map[2] = B["u"][1]

    # System
    sys = starry.System(pri, sec)

    # Compute the flux
    flux_model = sys.flux(t=t)

    # Track some values for plotting later
    pm.Deterministic("flux_model", flux_model)

    # Save our initial guess
    flux_model_guess = pmx.eval_in_model(flux_model)

    # The likelihood function assuming known Gaussian uncertainty
    pm.Normal("obs", mu=flux_model, sd=sigma_sd, observed=flux_real)


In [ ]:
with model:
    map_soln = pmx.optimize(start=model.test_point)

In [ ]:
map_soln

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(t, flux, "k.", alpha=0.3, ms=2, label="data")
plt.plot(t, flux_model_guess, "C1--", lw=1, alpha=0.5, label="Initial")
plt.plot(
    t, pmx.eval_in_model(flux_model, map_soln, model=model), "C1-", label="MAP", lw=1
)
plt.legend(fontsize=10, numpoints=5)
plt.xlabel("time [days]", fontsize=24)
plt.ylabel("relative flux", fontsize=24);

In [ ]:
with model:
    trace = pmx.sample(
        tune=1500, draws=1500, start=map_soln, cores=5, chains=4, target_accept=0.9, return_inferencedata=True,
    )

In [ ]:
var_names_A = ["A_m", "A_r", "A_inc", "u_A", 'A_amp']
var_names_B = ["B_m", "B_r", "B_porb", "B_inc", "B_t0", "B_ecc", "B_w", 'u_B', 'B_amp']
display(pm.summary(trace, var_names=var_names_A + var_names_B))

# Obtener el resumen
summary = pm.summary(trace, var_names=var_names_A + var_names_B)

# Guardar en un archivo CSV
summary.to_csv(Label+'_Summary.csv', sep=',')

In [ ]:
figure = corner(trace, var_names=var_names_A, show_titles=True,use_math_text=True,label_kwargs={"fontsize": 14})

for ax in figure.get_axes():
      ax.tick_params(axis='both', labelsize=12)


figure.savefig(Label+'_A.png',dpi=600,pad_inches=0.3,bbox_inches='tight')

In [ ]:
figure = corner(trace, var_names=var_names_B, show_titles=True,use_math_text=True,label_kwargs={"fontsize": 14})

for ax in figure.get_axes():
      ax.tick_params(axis='both', labelsize=12)


figure.savefig(Label+'_B.png',dpi=600, pad_inches=0.3, bbox_inches='tight')

In [ ]:
# Plot the results
curve = trace.posterior["flux_model"].values
q16, q50, q84 = np.percentile(curve, [16, 50, 84], axis=(0, 1))

Curva_Q50 = lk.LightCurve(time=t, flux=q50)
Curva_Q16 = lk.LightCurve(time=t, flux=q16)
Curva_Q84 = lk.LightCurve(time=t, flux=q84)

strm_1 = Curva_Q50.to_csv()
with open(Label+"_q50.csv", "w") as text_file:
    text_file.write(strm_1)

strm_2 = Curva_Q16.to_csv()
with open(Label+"_q16.csv", "w") as text_file:
    text_file.write(strm_2)

strm_3 = Curva_Q84.to_csv()
with open(Label+"_q84.csv", "w") as text_file:
    text_file.write(strm_2)

Curva_Q50

In [ ]:
fig, ax = plt.subplots(1, figsize=(18, 9))
ax.plot(t_real, flux_real, "k.", alpha=0.5, ms=10)
ax.plot(Curva_Q50['time'].value, Curva_Q50['flux'].value, lw=2)
# ax.plot(t_real, flux_real, ".k", ms=4, color= 'k' ,label="Sector 06")
ax.set_xlabel("time [days]", fontsize=24)
ax.set_ylabel("normalized flux", fontsize=24);

In [ ]:
import pandas as pd

inf_pos = {"A_m": trace.posterior["A_m"].values.reshape(-1), "A_r": trace.posterior["A_r"].values.reshape(-1),
           "A_inc": trace.posterior["A_inc"].values.reshape(-1), "A_amp": trace.posterior["A_amp"].values.reshape(-1),
           'A_u0':trace.posterior["u_A"][:,:,0].values.reshape(-1), 'A_u1':trace.posterior["u_A"][:,:,1].values.reshape(-1),
           "B_m": trace.posterior["B_m"].values.reshape(-1), "B_r": trace.posterior["B_r"].values.reshape(-1),
           "B_porb": trace.posterior["B_porb"].values.reshape(-1), "B_inc": trace.posterior["B_inc"].values.reshape(-1),
           "B_t0": trace.posterior["B_t0"].values.reshape(-1), "B_ecc": trace.posterior["B_ecc"].values.reshape(-1),
           "B_w": trace.posterior["B_w"].values.reshape(-1), "B_amp": trace.posterior["B_amp"].values.reshape(-1),
           'B_u0':trace.posterior["u_B"][:,:,0].values.reshape(-1), 'B_u1':trace.posterior["u_B"][:,:,1].values.reshape(-1),}
print(inf_pos["A_m"].shape)
df_pos = pd.DataFrame(inf_pos)

df_pos.to_csv(Label+"_Posteriors.csv", sep=',')

# Calcular el promedio de cada columna y mostrarlo en pantalla
column_means = df_pos.mean()
print(column_means)

In [ ]:
df_pos

In [ ]:
from astropy.io import ascii
starry.config.lazy = False
starry.config.quiet = True

data = ascii.read('data.txt', format='fast_no_header')
file = data[0][0]
path = data[2][0]

#system = file[-34:-15]
system = file[-32:-15]

print(system)

Pos = pd.read_csv(data[0][0])

Q16 = pd.read_csv(data[2][0])  
Q50 = pd.read_csv(data[3][0])  
Q84 = pd.read_csv(data[4][0])  


In [ ]:
def phase_fold(t, y, period, t0):
    phases=np.remainder(t-t0, period) / period
    phases=np.concatenate((phases,phases+1))
    y=np.concatenate((y,y))
    return phases, y

In [ ]:
CORNER_KWARGS = dict(
    smooth=0.5,
    label_kwargs=dict(fontsize=12),
    title_kwargs=dict(fontsize=10),
    quantiles=[0.16, 0.5, 0.84],
    plot_density=True,
    plot_datapoints=True,
    fill_contours=False,
    show_titles=True,
    max_n_ticks=3,
    title_fmt=".3f"
)

array_a = Pos[['A_m', 'A_r']].values

figure = corner(array_a,labels=['$M_A$', '$R_A$'], fig=plt.figure(figsize=(5.5,5.5)),
               use_math_text=True, **CORNER_KWARGS);

for ax in figure.get_axes():
      ax.tick_params(axis='both', labelsize=9)
      ax.tick_params(axis='x', which='both', pad=1)
      ax.tick_params(axis='y', which='both', pad=1)

# Change the number format of the tick labels for all axes
#ax.xaxis.set_major_formatter(ticker.StrMethodFormatter("{:.3f}"))
#ax.yaxis.set_major_formatter(ScalarFormatter(useOffset=False))

figure.savefig('Corner_'+system+'_A.png', dpi=300)

In [ ]:
CORNER_KWARGS = dict(
    smooth=0.5,
    label_kwargs=dict(fontsize=12),
    title_kwargs=dict(fontsize=10),
    quantiles=[0.16, 0.5, 0.84],
    plot_density=True,
    plot_datapoints=True,
    fill_contours=False,
    show_titles=True,
    max_n_ticks=3,
    title_fmt=".3f"
)

array_B = Pos[['B_m', 'B_r', 'B_inc', 'B_ecc', 'B_w']].values

G       = 6.6742e-11
M_sun   = 1.99e30
sec_day = 86400
AU      = 150e9

mass_1 = Pos["A_m"].values
P = Pos["B_porb"].values*sec_day

a = ((G*(mass_1 + array_B[:,0])*M_sun*P**2)/(4*np.pi**2))**(1/3)/AU

array_B = np.column_stack((array_B,a))

new_array = array_B[:,0]
new_array = np.column_stack((new_array,array_B[:,1:3]))
new_array = np.column_stack((new_array,a))
new_array = np.column_stack((new_array,array_B[:,3:5]))

figure = corner(new_array,labels=['$M$', '$R$', '$i$', '$a$', '$e$', '$\omega$'], fig=plt.figure(figsize=(8.5,8.5)),
               use_math_text=True, **CORNER_KWARGS);

for ax in figure.get_axes():
      ax.tick_params(axis='both', labelsize=9)
      ax.tick_params(axis='x', which='both', pad=1)
      ax.tick_params(axis='y', which='both', pad=1)

# Change the number format of the tick labels for all axes
#ax.xaxis.set_major_formatter(ticker.StrMethodFormatter("{:.3f}"))
#ax.yaxis.set_major_formatter(ScalarFormatter(useOffset=False))

figure.savefig('Corner_'+system+'_B.png', dpi=300)

In [ ]:
# Get the posterior median orbital parameters
M_a_16, M_a_50, M_a_84         = np.percentile(Pos['A_m'], [16, 50, 84])
R_a_16, R_a_50, R_a_84         = np.percentile(Pos['A_r'], [16, 50, 84])
i_a_16, i_a_50, i_a_84         = np.percentile(Pos['A_inc'], [16, 50, 84])
amp_a_16, amp_a_50, amp_a_84   = np.percentile(Pos['A_amp'], [16, 50, 84])
u0_a_16, u0_a_50, u0_a_84      = np.percentile(Pos['A_u0'], [16, 50, 84])
u1_a_16, u1_a_50, u1_a_84      = np.percentile(Pos['A_u1'], [16, 50, 84])

M_b_16, M_b_50, M_b_84         = np.percentile(Pos['B_m'], [16, 50, 84])
R_b_16, R_b_50, R_b_84         = np.percentile(Pos['B_r'], [16, 50, 84])
i_b_16, i_b_50, i_b_84         = np.percentile(Pos['B_inc'], [16, 50, 84])
p_b_16, p_b_50, p_b_84         = np.percentile(Pos['B_porb'], [16, 50, 84])
t0_b_16, t0_b_50, t0_b_84      = np.percentile(Pos['B_t0'], [16, 50, 84])
e_b_16, e_b_50, e_b_84         = np.percentile(Pos['B_ecc'], [16, 50, 84])
w_b_16, w_b_50, w_b_84         = np.percentile(Pos['B_w'], [16, 50, 84])
amp_b_16, amp_b_50, amp_b_84   = np.percentile(Pos['B_amp'], [16, 50, 84])
u0_b_16, u0_b_50, u0_b_84      = np.percentile(Pos['B_u0'], [16, 50, 84])
u1_b_16, u1_b_50, u1_b_84      = np.percentile(Pos['B_u1'], [16, 50, 84])

a_b_16, a_b_50, a_b_84         = np.percentile(a, [16, 50, 84])

print(amp_a_50, amp_b_50)

In [ ]:
import dataframe_image as dfi

data = {
    'Parameter' : ['M_a', 'R_a', 'u0_a', 'u1_a', 'M_b', 'R_b', 'Inc', 'Porb', 'a', 't0', 'ecc', 'w', 'u0_b', 'u1_b'],
    'Median'    : [M_a_50, R_a_50, u0_a_50, u1_a_50, M_b_50, R_b_50, i_b_50, p_b_50, a_b_50, t0_b_50, e_b_50, w_b_50,
                  u0_b_50, u1_b_50],
    'err_Q16'   : [M_a_16, R_a_16, u0_a_16, u1_a_16, M_b_16, R_b_16, i_b_16, p_b_16, a_b_16, t0_b_16, e_b_16, w_b_16,
                  u0_b_16, u1_b_16],
    'err_Q86'   :[M_a_84, R_a_84, u0_a_84, u1_a_84, M_b_84, R_b_84, i_b_84, p_b_84, a_b_84, t0_b_84, e_b_84, w_b_84,
                  u0_b_84, u1_b_84],
}

Final_info_B = pd.DataFrame(data)
display(Final_info_B)

dfi.export(Final_info_B, 'Table_'+system+'.png', table_conversion='matplotlib')

In [ ]:
A = starry.Primary(starry.Map(ydeg=3, udeg=2, amp=amp_a_50), r=R_a_50, m=M_a_50)
A.map[1:] = [u0_a_50, u1_a_50]

A.map.show()

In [ ]:
b = starry.Secondary(
    starry.Map(ydeg=3, udeg=2, inc=i_b_50, amp=amp_b_50),
    r=R_b_50,
    m=M_b_50,
    porb=p_b_50,
    prot=1.0,
    t0=t0_b_50,
    inc=i_b_50,
    ecc=e_b_50,
    w=w_b_50,
)
b.map[1:] = [u0_b_50, u1_b_50]

b.map.show()

In [ ]:
sys = starry.System(A, b)
sys

In [ ]:
npts = 200
t = np.linspace(t0_b_50 - p_b_50/2, t0_b_50 + p_b_50/2, npts)
x, y, z = sys.position(t)
x = x - x[0]
y = y - y[0]
z = z - z[0]


In [ ]:
res = 300
theta_sec = [360.0 / sec.prot * (t - sec.t0) - sec.theta0 for sec in sys.secondaries]
img = np.array(
    [np.tile(sys.primary.map.render(res=300), (npts, 1, 1))]
    + [
        sec.map.render(theta=theta_sec[i], res=res)
        for i, sec in enumerate(sys.secondaries)
    ]
)

In [ ]:
flux = sys.flux(t)

t_fold = ((t - t0_b_50)/p_b_50)

In [ ]:
im0 = sys.show(t, window_pad=5, figsize=(5, 5))

In [ ]:
# Set up the plot
fig, ax = plt.subplots(1, figsize=(6.5, 7))
ax_xz = fig.add_axes([0.275, 0.8, 0.2, 0.2])
ax_xz.annotate(
    "Top", fontsize=12, xy=(0, 0), xycoords="axes fraction", ha="left", va="bottom"
)
ax_zy = fig.add_axes([0.525, 0.8, 0.2, 0.2])
ax_zy.annotate(
    "Side", fontsize=12, xy=(0, 0), xycoords="axes fraction", ha="left", va="bottom"
)
ax_lc = fig.add_axes([0.125, 0.1, 0.775, 0.2])

xz = [None] + [None for sec in sys.secondaries]
xy = [None] + [None for sec in sys.secondaries]
zy = [None] + [None for sec in sys.secondaries]
circ = [None] + [None for sec in sys.secondaries]
maps = [sys.primary.map] + [sec.map for sec in sys.secondaries]
radii = np.array([sys.primary.r] + [sec.r for sec in sys.secondaries])

for axis, arrs in zip([ax, ax_xz, ax_zy], [(x, y), (x, z), (z, y)]):
    axis.axis("off")
    R = 1.2 * max(-np.min(arrs), np.max(arrs))
    axis.set_xlim(-R, R)
    axis.set_ylim(-R, R)

# Plot the light curve
ax_lc.plot(t_fold, flux, "k-")
(lc,) = ax_lc.plot(t_fold[0], flux[0], "o", color="k")
ax_lc.axis("on")
ax_lc.set_xlim(t_fold[0], t_fold[-1])
ax_lc.set_xlabel('Folded Phase')
ax_lc.set_ylabel('Normalised Flux')

# Plot the first frame
for i, xi, yi, zi, map, r in zip(range(1 + len(sys.secondaries)), x, y, z, maps, radii):

    # Orbit outlines
    ax_xz.plot(xi, zi)
    ax_zy.plot(zi, yi)

    # Body positions
    xz[i] = ax_xz.scatter(xi[0], zi[0])
    zy[i] = ax_zy.scatter(zi[0], yi[0])

    # Maps
    extent = np.array([xi[0], xi[0], yi[0], yi[0]]) + np.array([-1, 1, -1, 1]) * r
    xy[i] = ax.imshow(
        img[i,0],
        origin="lower",
        cmap="plasma",
        extent=extent,
        clip_on=False,
        zorder=zi[0],
    )
    circ[i] = plt.Circle(
        (xi[0], yi[0]), r, color="k", fill=False, zorder=zi[0] + 1e-3, lw=3
    )
    ax.add_artist(circ[i])

# Animation
def updatefig(k):
    for i, xi, yi, zi, map, r in zip(
        range(1 + len(sys.secondaries)), x, y, z, maps, radii
    ):
        xz[i].set_offsets((xi[k], zi[k]))
        zy[i].set_offsets((zi[k], yi[k]))
        xy[i].set_extent(
            np.array([xi[k], xi[k], yi[k], yi[k]]) + np.array([-1, 1, -1, 1]) * r
        )
        xy[i].set_zorder(zi[k])
        xy[i].set_data(img[i, k])
        circ[i].center = (xi[k], yi[k])
        circ[i].set_zorder(zi[k] + 1e-3)
    lc.set_xdata(t_fold[k])
    lc.set_ydata(flux[k])
    return xz + xy + zy + circ + [lc]


ani = animation.FuncAnimation(fig, updatefig, interval=30, blit=False, frames=len(t))
plt.close()
display(HTML(ani.to_html5_video()))

writergif = animation.FFMpegWriter(fps=20) 
ani.save('system.mp4', writer=writergif)

## Gráfica de curvas de luz

In [ ]:

time = np.asarray(data["time"])
flux = np.asarray(data["flux"])
flat_flux = flat_flux
trend_flux = trend_flux

In [ ]:
po = p_b_50
to1 = t0_b_50

t_fold_q, q50 = phase_fold(Q50['time'].values, Q50['flux'].values, po, to1)
_, q16 = phase_fold(Q16['time'].values, Q16['flux'].values, po, to1)
_, q84 = phase_fold(Q84['time'].values, Q84['flux'].values, po, to1)

inds = t_fold_q.argsort()
t_fold_q_sorted = t_fold_q[inds]
q50_sorted = q50[inds]
q16_sorted = q16[inds]
q84_sorted = q84[inds]

print(t_fold_q_sorted)

In [ ]:
Times_norm_1 = time
Flux_norm_1 = flat_flux
ph_norm, f1_norm = phase_fold(np.array(Times_norm_1), np.array(Flux_norm_1), po, to1)

plt.figure(figsize=(10,5))
plt.scatter(ph_norm-1, f1_norm, marker="o", color='C1', s=20, alpha=0.8, edgecolors='black', lw=0.4, label='Data')
plt.plot(t_fold_q_sorted-1, q50_sorted, color='darkblue', label='Best Fit')
plt.xlabel('Folded Phase')
plt.ylabel('Normalized Flux')
plt.title('The period of the object is: {:.4f} days'.format(p_b_50))
plt.xlim(-0.5, 0.5)
#plt.ylim(min(f1_norm)*0.99, max(f1_norm)*0.97)
plt.legend(loc='lower left')
plt.savefig(system+'.png',dpi=300)

In [ ]:
print(len(time), len(flux))

In [ ]:
from matplotlib.gridspec import GridSpec
import os
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 14})

# =========================
# Asegurar misma longitud
# =========================
n = min(len(time), len(flux), len(trend_flux), len(flat_flux))
time = time[:n]
flux = flux[:n]
trend_flux = trend_flux[:n]
flat_flux = flat_flux[:n]

# =========================
# FIGURA
# =========================
fig = plt.figure(figsize=(8,12), constrained_layout=True)
gs = GridSpec(3, 1, height_ratios=[1,1,1], hspace=0.06, figure=fig)

# =========================
# 1. ORIGINAL + TREND
# =========================
ax1 = fig.add_subplot(gs[0])

ax1.scatter(time, flux,
            s=10, c='black',
            label='Original')

ax1.plot(time, trend_flux,
         c='r', lw=2, ls='--',
         label='Trend')

ax1.set_ylabel("Flux")
ax1.set_ylim(np.min(flux)*0.995, np.max(flux)*1.006)
ax1.legend(loc='upper left')


# =========================
# 2. FLATTENED
# =========================
ax2 = fig.add_subplot(gs[1], sharex=ax1)

ax2.scatter(time, flat_flux,
            c='mediumblue', s=5,
            label='Flattened')

ax2.set_ylabel("Normalized Flux")
ax2.set_xlabel("Time [days]")
ax2.set_ylim(np.min(flat_flux)*0.995, np.max(flat_flux)*1.006)
ax2.legend(loc='upper left')


# =========================
# 3. PHASE FOLD
# =========================

# centrar fase correctamente 👀
phase_data = ph_norm - np.floor(ph_norm + 0.5)
phase_model = t_fold_q_sorted - np.floor(t_fold_q_sorted + 0.5)

# ordenar modelo para evitar artefactos
inds = np.argsort(phase_model)
phase_model = phase_model[inds]
q50_sorted = q50_sorted[inds]
q16_sorted = q16_sorted[inds]
q84_sorted = q84_sorted[inds]

ax3 = fig.add_subplot(gs[2])

# DATA
ax3.scatter(phase_data, f1_norm,
            s=20, color='C1',
            alpha=0.8,
            edgecolors='black', lw=0.4,
            label='Data')

# MODELO (sin línea rara 👀)
ax3.scatter(phase_model, q50_sorted,
            color='darkblue', s=5,
            label='Best Fit')

# POSTERIOR
ax3.fill_between(phase_model,
                 q16_sorted, q84_sorted,
                 alpha=0.3, color='darkblue',
                 label='Posterior')

ax3.set_xlabel('Phase')
ax3.set_ylabel('Normalized Flux')
ax3.set_xlim(-0.5, 0.5)

ax3.legend(loc='lower right')

ax3.text(-0.48, np.min(q50_sorted),
         f'$P$ = {po:.4f} days',
         fontsize=14)

# =========================
# SAVE
# =========================
filename = f"Full_{system.split('/')[-1]}.png"
plt.savefig(filename, dpi=300)

plt.show()

In [ ]:
# Get the posterior median orbital parameters
M_a_16, M_a_50, M_a_84         = np.percentile(Pos['A_m'], [16, 50, 84])
R_a_16, R_a_50, R_a_84         = np.percentile(Pos['A_r'], [16, 50, 84])
i_a_16, i_a_50, i_a_84         = np.percentile(Pos['A_inc'], [16, 50, 84])
amp_a_16, amp_a_50, amp_a_84   = np.percentile(Pos['A_amp'], [16, 50, 84])
u0_a_16, u0_a_50, u0_a_84      = np.percentile(Pos['A_u0'], [16, 50, 84])
u1_a_16, u1_a_50, u1_a_84      = np.percentile(Pos['A_u1'], [16, 50, 84])

M_b_16, M_b_50, M_b_84         = np.percentile(Pos['B_m'], [16, 50, 84])
R_b_16, R_b_50, R_b_84         = np.percentile(Pos['B_r'], [16, 50, 84])
i_b_16, i_b_50, i_b_84         = np.percentile(Pos['B_inc'], [16, 50, 84])
p_b_16, p_b_50, p_b_84         = np.percentile(Pos['B_porb'], [16, 50, 84])
t0_b_16, t0_b_50, t0_b_84      = np.percentile(Pos['B_t0'], [16, 50, 84])
e_b_16, e_b_50, e_b_84         = np.percentile(Pos['B_ecc'], [16, 50, 84])
w_b_16, w_b_50, w_b_84         = np.percentile(Pos['B_w'], [16, 50, 84])
amp_b_16, amp_b_50, amp_b_84   = np.percentile(Pos['B_amp'], [16, 50, 84])
u0_b_16, u0_b_50, u0_b_84      = np.percentile(Pos['B_u0'], [16, 50, 84])
u1_b_16, u1_b_50, u1_b_84      = np.percentile(Pos['B_u1'], [16, 50, 84])

a_b_16, a_b_50, a_b_84         = np.percentile(a, [16, 50, 84])

In [ ]:
data_A = {
    'Variable' : ['$M_A$', '$R_A$', '$u_{0,A}$', '$u_{1,A}$'],
    'Median'    : [M_a_50, R_a_50, u0_a_50, u1_a_50],
    'Q16'       : [M_a_16-M_a_50, R_a_16-R_a_50, u0_a_16-u0_a_50, u1_a_16-u1_a_50],
    'Q86'       : [M_a_84-M_a_50, R_a_84-R_a_50, u0_a_84-u0_a_50, u1_a_84-u1_a_50],
}

Final_A = pd.DataFrame(data_A)
display(Final_A.style.hide())



data_B = {
    'Variable' : ['$M_B$', '$R_B$', '$i$', '$P_o$', '$a$', '$t_0$', '$e$', '$\omega$', '$u_{0,B}$', '$u_{1,B}$'],
    'Median'    : [M_b_50, R_b_50, i_b_50, p_b_50, a_b_50, t0_b_50, e_b_50, w_b_50, u0_b_50, u1_b_50],
    'Q16'   : [M_b_16-M_b_50, R_b_16-R_b_50, i_b_16-i_b_50, p_b_16-p_b_50, 
                   a_b_16-a_b_50, t0_b_16-t0_b_50, e_b_16-e_b_50, w_b_16-w_b_50, u0_b_16-u0_b_50, u1_b_16-u1_b_50],
    'Q86'   : [M_b_84-M_b_50, R_b_84-R_b_50, i_b_84-i_b_50, p_b_84-p_b_50, 
                   a_b_84-a_b_50, t0_b_84-t0_b_50, e_b_84-e_b_50, w_b_84-w_b_50, u0_b_84-u0_b_50, u1_b_84-u1_b_50],
}

Final_B = pd.DataFrame(data_B)
display(Final_B.style.hide())


In [ ]:
Fl_Det = flux/trend_flux

fig = plt.figure(figsize=(12,5), dpi=300)
#plt.plot(Time, Flux, ".k")
#plt.plot(Times, Fl_Det, c='b')
plt.scatter(time, Fl_Det, c='b',s = 10)#s tamaño de punto 
plt.ylabel("Flujo")
plt.xlabel("Tiempo")
plt.rc('font', size=12)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.timeseries import LombScargle
from scipy.stats import median_abs_deviation

P_orb = 2.1
t0 = to1   # tiempo de tránsito (ya lo tienes)

phase_orb = ((time - t0) / P_orb) % 1
phase_orb = phase_orb - 0.5

mask = (np.abs(phase_orb) > 0.15) & (np.abs(np.abs(phase_orb) - 0.5) > 0.1)

time_clean = time[mask]
flux_clean = (flux / trend_flux)[mask]   

plt.figure(figsize=(10,4))

plt.scatter(time, flux/trend_flux,
            s=5, color='gray', alpha=0.3,
            label='Original')

plt.scatter(time_clean, flux_clean,
            s=5, color='blue',
            label='Sin eclipses')

plt.xlabel("Time [days]")
plt.ylabel("Normalized Flux")
plt.legend()

plt.title("Curva sin eclipses")
plt.tight_layout()
plt.show()

time_cont = np.arange(len(time_clean))

plt.figure(figsize=(10,4))

plt.scatter(time_cont, flux_clean,
            s=5, color='blue')

plt.xlabel("Time index (continuous)")
plt.ylabel("Normalized Flux")
plt.title("Curva sin eclipses (continua)")

plt.tight_layout()
plt.show()


In [ ]:

# =========================
# 1. NORMALIZAR
# =========================
flux_ls = flux_clean / np.median(flux_clean)
flux_ls = flux_ls - np.mean(flux_ls)

# =========================
# 2. LOMB–SCARGLE
# =========================
ls = LombScargle(time_clean, flux_ls)

frequency, power = ls.autopower(
    minimum_frequency=0.1,   # periodos largos (~10 d)
    maximum_frequency=5.0    # periodos cortos (~0.2 d)
)

period = 1 / frequency

# =========================
# 3. MEJOR PERIODO
# =========================
best_period = period[np.argmax(power)]

print(f"Periodo dominante: {best_period:.4f} días")
print(f"Posible rotación: {2*best_period:.4f} días (si es P/2)")

# =========================
# 4. PLOT PERIODOGRAMA
# =========================
plt.figure(figsize=(10,4))

plt.plot(period, power, 'k-')

# marcar P/2
plt.axvline(best_period,
            color='red', ls='--',
            label=f'{best_period:.3f} d (P/2?)')

# marcar P
plt.axvline(2*best_period,
            color='blue', ls='--',
            label=f'{2*best_period:.3f} d (P?)')

plt.xlabel("Period [days]")
plt.ylabel("Power")
plt.xlim(0, 5)
plt.legend()

plt.title("Lomb-Scargle (sin eclipses)")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================
# periodo (elige cuál quieres)
# =========================
P_plot = 2 * best_period   # o best_period si quieres probar P/2

# =========================
# t0 (puedes usar orbital o rotacional)
# =========================
t0_plot = t0_rot   # o to1 si quieres eclipse centrado

# =========================
# fase en [0,1]
# =========================
phase = ((time_clean - t0_plot) / P_plot) % 1

# =========================
# ordenar (IMPORTANTE 👀)
# =========================
inds = np.argsort(phase)
phase = phase[inds]
flux_plot = flux_clean[inds]

# =========================
# plot estilo transit
# =========================
plt.figure(figsize=(6,4))

plt.scatter(phase, flux_plot,
            s=10, color='red', alpha=0.4)

plt.xlabel("Phase [days]")
plt.ylabel("Flux Corrected")

plt.xlim(0, 1)

plt.tight_layout()
plt.show()